# 4 Searching architectures instead of guessing them

A training run now costs about two seconds. That single fact makes hand-tuning
indefensible — not inefficient, *indefensible*, because you can no longer claim you picked
the width by intuition when checking twenty widths would have cost a coffee.

So we stop guessing. This notebook hands the architecture space to a search algorithm and
reads what comes back, honestly, including the parts that are disappointing.

**We switch datasets here, on purpose.** Notebooks 01–03 used EuroSAT, where the point was
that the approach works at all. But EuroSAT's three encoders land within three points of
each other, so a search there is flat on every axis — including the one that is supposed
to matter. DTD (textures) spreads those same encoders across fourteen points and leaves
about twenty points of headroom. Run your search where the result is legible.

## Setup: the caches, the loaders and a timed trainer

Nothing to study in the next two cells. They load the DTD caches of four encoders, make
`loaders(encoder)` for the train and test split, and define `run(model, ...)`, which trains
a model and returns its test accuracy and the seconds it spent training.

In [ ]:
import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import Concatenate2D, GaussianNoise, NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data import Collate, CollateParallel, OneHot
from vectormesh.training import Trainer, TrainerSettings

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
# a search shows one progress bar; Optuna's line per trial stays off
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
SEEDS = param(5, test=2)

ENCODERS = ["dinov2-large", "dinov2-base", "dinov2-small", "resnet-18"]
caches = {
    enc: {
        split: VectorCache.from_hub(f"pttrn-io/dtd-{enc}", split=split)
        for split in ("train", "test")
    }
    for enc in ENCODERS
}

N_CLASSES = caches["dinov2-small"]["train"].num_classes
DIM = {enc: caches[enc]["train"].dim for enc in ENCODERS}

print(
    f"DTD: {len(caches['dinov2-small']['train'])} train / {len(caches['dinov2-small']['test'])} test, {N_CLASSES} classes"
)
print("encoder widths:", DIM)

In [ ]:
DEFAULT_ONEHOT = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
DEFAULT_COLLATE = Collate(
    embedding_col="embed", target_col="onehot", padder=torch.stack
)


def loaders(
    encoder: str,
    batch_size: int = 64,
    onehot: OneHot | None = None,
    collate: Collate | None = None,
):
    # onehot/collate are arguments, not a closure over a module global -- loaders(name) still
    # works via the defaults below, but a caller with a different label scheme is not stuck.
    onehot = onehot or DEFAULT_ONEHOT
    collate = collate or DEFAULT_COLLATE
    return [
        DataLoader(
            caches[encoder][split].dataset.map(onehot),
            batch_size=batch_size,
            shuffle=(split == "train"),
            collate_fn=collate,
        )
        for split in ("train", "test")
    ]


class TimedTrainer(Trainer):
    """A Trainer that adds up the seconds spent in training steps; evaluation after every
    epoch is not counted."""

    train_seconds = 0.0

    def trainbatches(self) -> float:
        start = time.perf_counter()
        loss = super().trainbatches()
        self.train_seconds += time.perf_counter() - start
        return loss


def run(
    model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS
) -> tuple[float, float]:
    """Train `model` and return (test accuracy, seconds spent in training steps)."""
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(train_dl),
        valid_steps=len(test_dl),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=train_dl,
        validdataloader=test_dl,
        progress=False,
    )
    return trainer.loop().metric_dict["Accuracy"], trainer.train_seconds

## 4.0 The baseline, before anything else

Notebook 03 said it and it is worth repeating as a rule: **a search without a baseline is
unreadable.** If you do not know what the two-line model scores, you cannot tell whether
the search's best trial is a discovery or a rounding error.

So establish it first, in two steps. Start with a floor: 1-nearest-neighbour, which trains
nothing at all — every test vector takes the label of its closest training vector. Then
the two-line model itself, with a seed spread, because a single run is an anecdote.

### Every run goes to an Optuna study, starting now

Every number this notebook produces is kept in Optuna's store, `artefacts/optuna/studies.db`
at the root of the repository, and read in optuna-dashboard. The floor, the baseline and the
fusion arm are trials of one study, `lesson1-search-runs`; the search is a study of its own,
`lesson1-search`. The dashboard shows them as rows of one table, sortable on the value, and the
question "did the search beat the baseline?" becomes a sort, not a scroll back through
cell outputs.

The floor, the baseline and the fusion arm are runs that are given their params, and they
are written through `StudyLog.log` from `hpo.record.runs`, the same class notebook 05 uses, so
every run has the same shape:

- a **`role`** user attr: `baseline`, `trial` (one search trial), `hypothesis` (the fusion arm),
  or `ablation` (the 1-NN floor: the baseline with the training taken out);
- the **params** that define it (`encoder`, `head`, `n_hidden_layers`, `hidden_dim`,
  `sigma`, `lr`, `epochs` — whichever apply);
- the **value**: the accuracy, the mean over the repeats of a run that was repeated over
  seeds, with `accuracy_std` and `n_repeats` next to it and every repeat as an
  intermediate value;
- the **architecture**: `n_params`, a one-line `architecture` user attr, and the layer table
  with the tensor shape after every layer, as the trial's note and as the artifact
  `architecture.json`.

Every trial also carries `session`, the time you started this run of the notebook, which keeps
a rerun from getting mixed up with the last one.

In [ ]:
from hpo.measure import Architecture, Scalars
from hpo.record import SESSION, StudyLog

# Optuna needs to know in advance which values a text param can take. These are the
# values the runs of this notebook log for `dataset`, `encoder` and `head`: four
# encoders and the fused pair, and the four kinds of head this notebook trains.
CHOICES = {
    "dataset": ["dtd"],
    "encoder": [*ENCODERS, "dinov2-small+resnet-18"],
    "head": ["1-nearest-neighbour", "linear", "mlp", "concat+linear"],
}
# the Optuna study the floor, the baseline and the fusion arm are written to
runs = StudyLog.runs(name="lesson1-search-runs", choices=CHOICES)

### The floor: 1-nearest-neighbour, nothing trained

Every test vector takes the label of the training vector closest to it. No weights, no
training: what this scores is what the encoder's vector space gives you for free.

In [ ]:
from plotting import result_bars

MODEL = "dinov2-small"  # the encoder the fusion arm in 4.3 is measured against
tr = caches[MODEL]["train"]
te = caches[MODEL]["test"]

# cache["col"] is a lazy Column; the [:] materialises it into one tensor
Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]

A = torch.nn.functional.normalize(Xtr, dim=1)  # cosine similarity = dot product
B = torch.nn.functional.normalize(Xte, dim=1)  # once both sides are unit length
neighbour = (B @ A.T).argmax(dim=1)  # (n_test, n_train) -> index of the closest

CHANCE = 1 / N_CLASSES
KNN_FLOOR = float((ytr[neighbour] == yte).float().mean())

runs.log(
    name="1nn-floor",
    role="ablation",
    results=Scalars(accuracy=[KNN_FLOOR]),
    dataset="dtd",
    params={
        "dataset": "dtd",
        "encoder": MODEL,
        "embed_dim": DIM[MODEL],
        "head": "1-nearest-neighbour",
    },
)

result_bars(
    {f"chance (1 in {N_CLASSES})": CHANCE, "1-nearest-neighbour": KNN_FLOOR},
    title=f"{MODEL} on DTD, zero parameters trained",
)
plt.show()

### The baseline: one linear layer, trained a few times

Now the simplest model that is trained: one `Linear(384, 47)` on the same vectors. It is
trained `SEEDS` times. A seed sets the random weights a run starts from and the order of
its batches, so every seed ends at a slightly different test accuracy. The bar is the mean
of those accuracies, the error bar their standard deviation, and every dot is one seed.

In [ ]:
train_dl, test_dl = loaders(MODEL)

# torch.manual_seed(seed) makes a run reproducible: seed 0 gives the same accuracy on
# every rerun, and the difference between the seeds is the spread between runs
accuracy_per_seed, seconds = [], []
for seed in range(SEEDS):
    torch.manual_seed(seed)
    baseline = torch.nn.Linear(DIM[MODEL], N_CLASSES)
    accuracy, spent = run(baseline, train_dl, test_dl)
    accuracy_per_seed.append(accuracy)
    seconds.append(spent)

BASELINE = float(np.mean(accuracy_per_seed))
BASELINE_STD = float(np.std(accuracy_per_seed))

# one trial for all seeds: the value is the mean, the spread is logged next to it
runs.log(
    name="baseline",
    role="baseline",
    results=Scalars(accuracy=accuracy_per_seed, seconds=float(np.mean(seconds))),
    dataset="dtd",
    params={
        "dataset": "dtd",
        "encoder": MODEL,
        "embed_dim": DIM[MODEL],
        "head": "linear",
        "n_hidden_layers": 0,
        "sigma": 0.0,
        "lr": 1e-3,
        "epochs": EPOCHS,
    },
    architecture=Architecture.of(baseline, torch.zeros(2, DIM[MODEL])),
)

result_bars(
    {
        f"chance (1 in {N_CLASSES})": CHANCE,
        "1-nearest-neighbour": KNN_FLOOR,
        f"linear head, {SEEDS} seeds": accuracy_per_seed,
    },
    title=f"the reference every later number is read against ({MODEL} on DTD)",
    highlight=f"linear head, {SEEDS} seeds",
)
plt.show()

The error bar is the number people skip, and it is the one that decides what counts as a
win. A later result that beats the baseline's mean by less than about twice that spread
has not beaten anything.

**Open optuna-dashboard now**, before the search adds two dozen more trials. In a terminal,
from the root of the repository:

```bash
make optuna-dashboard
```

and open <http://127.0.0.1:8088>. Pick the study `lesson1-search-runs`. You should see a
`1nn-floor` trial and a `baseline` trial, each with its accuracy as the value; the baseline
also has `accuracy_std` among its user attrs and one intermediate value per seed. Open the
baseline trial: its note lists every layer with its input and output shape, and
`architecture.json` is among its artifacts. From §4.1 on the search trials land in a study
of their own, `lesson1-search`, which the dashboard lists next to this one.

## 4.1 The search space

A search space is the list of things we are willing to vary, each with the values it may
take. Ours has five. For each one: what it is, and why it is worth trying.

- **`encoder`: the pretrained model the vectors come from.** It is the base everything
  else is built on, and §4.0 showed how good a start it is: with nothing trained,
  1-nearest-neighbour is already far above chance. Notebook 01 measured that the encoders
  are fourteen points apart on this dataset, so the search may choose between four.
  ([goal 1.5](../../presentations/leerdoelen/les1.html#g1-5))
- **`n_hidden_layers`: how deep the head is**, 0, 1 or 2 hidden layers. 0 is the baseline,
  one `Linear`. Every hidden layer, with the nonlinearity after it, lets the head learn a
  more complex function of the vectors. That has a price: more weights take longer to
  train and need more data to be learned well, and you do not always have that time and
  that data. ([goal 1.6](../../presentations/leerdoelen/les1.html#g1-6))
- **`hidden_dim`: how wide each hidden layer is**, 128 or 512 units. Width adds complexity
  just as depth does. The universal approximation theorem from the slides says that one
  hidden layer can approximate any continuous function *if it is wide enough*. It does not
  say how wide, so we try two widths. The price is the same as for depth: more weights.
  Without hidden layers there is nothing to be wide, so the model ignores `hidden_dim`
  when `n_hidden_layers = 0`. ([goal 1.6](../../presentations/leerdoelen/les1.html#g1-6))
- **`sigma`: how much Gaussian noise is added to the vectors during training.** A head can
  memorise its training vectors: perfect on the training set, worse on images it has not
  seen. With a little random noise on every vector the head never sees exactly the same
  input twice, so it cannot lean on the exact value of one number and has to learn what
  holds for everything close to a vector. That is regularisation. With images you would
  flip or crop the picture for the same reason; the pixels are gone here, so the noise
  goes on the vectors. At test time the noise is off. ([goal 1.9](../../presentations/leerdoelen/les1.html#g1-9))
- **`lr`: the learning rate, the size of the step the optimizer takes** every time it
  updates the weights. The gradient says in which direction the loss goes down, the
  learning rate says how far to go in that direction. Too small, and the model has barely
  moved when the epochs are over. Too large, and it steps over the good weights and the
  loss jumps around. We do not know in advance whether 0.0001 or 0.001 is the right size,
  so every *order of magnitude* should get as many trials: as many between 0.0001 and
  0.0003 as between 0.001 and 0.003. That is what sampling on a log scale does, and "What
  the search will sample" below shows it in a picture. ([goal 1.7](../../presentations/leerdoelen/les1.html#g1-7))

`build_model` turns one such set of choices into a model. With 0 hidden layers it is a
plain `nn.Linear`. With 1 or 2 it is one call of vectormesh's `NeuralNet`:
`NeuralNet([D, H, H], 47)` is `Linear(D, H) → GELU → Linear(H, H) → GELU → Linear(H, 47)`.
It is one `NeuralNet` with the whole chain of widths, not two stacked, because stacking
puts a bare `Linear → Linear` in the middle, and notebook 02 showed that two linear layers
without a nonlinearity between them are one linear layer. The search's trials use
`build_model`, and so do the diagrams below, so what you see drawn is what gets trained.

In [ ]:
from hpo.measure import Curves
from hpo.record import RunRecord
from hpo.search import Choice, Float, Space


def build_model(config: dict, dim: int, n_classes: int) -> nn.Module:
    n_hidden = config["n_hidden_layers"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)  # the baseline
    else:
        head = NeuralNet(
            [dim, *[config["hidden_dim"]] * n_hidden], n_classes
        )  # GELU between every Linear
    return Serial([GaussianNoise(sigma=config["sigma"]), head])


param_space = Space(
    {
        "encoder": Choice(ENCODERS),
        "n_hidden_layers": Choice([0, 1, 2]),  # 0 = the baseline's single Linear
        "hidden_dim": Choice([128, 512]),  # ignored when n_hidden_layers = 0
        "sigma": Float(0.0, 0.4),
        "lr": Float(1e-4, 3e-3, log=True),
        "epochs": EPOCHS,
    }
)

### What each option changes in the model

Every model in the space has the same skeleton: a batch of `B` cached vectors of width `D`
goes in, noise is added, a head maps it to 47 class scores, and the loss compares those
with the labels. The drawing below is made by `trace`, which pushes a dummy batch through
the real `build_model(...)` output and writes the shape of every tensor on the arrow it
travels along. Read the shapes: the batch dimension `B` never changes, the noise leaves the
shape alone, and every `Linear(a → b)` turns `(B, a)` into `(B, b)`.

In [ ]:
from diagrams import Block, draw_architectures, plot_search_space, sample_config, trace

LOSS = Block("CrossEntropyLoss\nvs labels", kind="loss")


def diagram(config: dict) -> list:
    """Input cache, the traced model, the loss: one row of a drawing."""
    encoder, dim = config["encoder"], DIM[config["encoder"]]
    model = build_model(config, dim, N_CLASSES)
    return [
        Block(f"{encoder}\ncache", f"(B, {dim})", "data"),
        *trace(model, torch.zeros(2, dim)),
        LOSS,
    ]


example = {"encoder": "dinov2-small", "hidden_dim": 512, "sigma": 0.1}
rows = {
    f"n_hidden_layers = {n}": diagram({**example, "n_hidden_layers": n})
    for n in (0, 1, 2)
}
note = "\n".join(
    [
        "encoder      sets D, the input width: "
        + ", ".join(f"{e} {d}" for e, d in DIM.items()),
        "hidden_dim   sets H, the width of every hidden layer: 128 or 512 (drawn: 512)",
        "sigma        sets the noise strength; at test time the noise block passes its input through unchanged",
        "lr           is not in the drawing: it sets the optimizer's step size, not the model",
    ]
)
draw_architectures(rows, note=note)
plt.show()

### What the search will sample

`param_space` is not a list of models, it is a set of distributions, one per axis. A trial
draws one value from each. Here are a few thousand draws per axis, before a single model
is trained, with one trial's draw marked in red:

- the three `choice` axes are flat: every option is equally likely;
- `sigma` is flat between 0 and 0.4;
- `lr` is flat **on a log axis**: as many trials between 1e-4 and 3e-4 as between 1e-3
  and 3e-3. The last panel shows the same draws on a linear axis, where they pile up at
  the low end. That is the point: a step from 1e-4 to 2e-4 doubles the learning rate, a
  step from 2.9e-3 to 3e-3 barely changes it, so the small values deserve as many trials.

Below the panels is the model that one red trial would train.

In [ ]:
trial = sample_config(param_space, seed=6)
print(
    "one trial:",
    {k: round(v, 5) if isinstance(v, float) else v for k, v in trial.items()},
)

plot_search_space(param_space, trial=trial)
plt.show()

draw_architectures({"the model this trial trains": diagram(trial)})
plt.show()

We use **TPE** (`optuna.samplers.TPESampler()`) to pick the trials. The first ten are drawn
from the distributions above; after that TPE models which regions produced good scores and
samples where the expected improvement is highest. A grid search would spend most of its
budget on combinations it already knows are bad. Lesson 3 opens up the algorithms themselves;
here it is one tool, used properly. A study that exists already gets the new trials added to
it, so on a rerun TPE starts from what the earlier trials found; the tables and plots below
read only the trials of this run.

**On `hidden_dim` being meaningless at `n_hidden_layers = 0`:** Optuna *can* express a
genuinely conditional space. In a hand-written objective, `trial.suggest_int("hidden_dim", ...)`
sits inside an `if n_hidden_layers > 0:`, so the sampler never models a `hidden_dim` that
cannot exist. That is worth knowing, and worth reaching for once a search space has real
nesting. It is not worth building here: `Space.draw` suggests every axis of the flat space, with the
unused one simply ignored by `build_model`, and that costs nothing at this trial budget. Reach
for the `if` when flat axes start wasting a noticeable fraction of the budget, not by default.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """What the study calls once per trial: draw a config from `param_space`, train it,
    report the accuracy after every epoch, and return the last one."""
    config = param_space.draw(trial)
    encoder = config["encoder"]
    dim = DIM[encoder]
    tr, te = loaders(encoder)

    torch.manual_seed(0)
    model = build_model(config, dim, N_CLASSES)
    curves = Curves()

    # report every epoch, not only the last: the trial's learning curve reaches the store
    def to_optuna(epoch, train_loss, test_loss, metric_dict):
        curves.add(train_loss, test_loss, None)
        trial.report(metric_dict["Accuracy"], epoch)

    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),  # never created unless earlystop saves
        train_steps=len(tr),
        valid_steps=len(te),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=te,
        progress=False,  # the study's own bar already shows progress across trials
        reporters=[to_optuna],
    )
    accuracy = trainer.loop().metric_dict["Accuracy"]
    study.record(
        trial,
        RunRecord(
            role="trial",
            dataset="dtd",
            config=config,
            fixed=param_space.fixed(),
            train_seconds=trainer.train_seconds,
            curves=curves,
            extra={
                "embed_dim": dim,
                "head": "linear" if config["n_hidden_layers"] == 0 else "mlp",
            },
        ),
        Architecture.of(model, torch.zeros(2, dim)),
    )
    return accuracy

In [ ]:
study = StudyLog.search(
    name="lesson1-search",
    space=param_space,
    sampler=optuna.samplers.TPESampler(),
    attrs={"origin": "04_search"},
)

start = time.time()
study.optimize(objective, n_trials=N_TRIALS)
print(
    f"{N_TRIALS} trials in {time.time() - start:.0f}s; the study {study.name!r} now holds {len(study.trials())}"
)

Each trial was written to the study `lesson1-search` while it ran: its params, its accuracy
after every epoch, its loss curves, `train_seconds`, and the architecture it trained.

§4.2 reads the result in three pictures. optuna-dashboard has the same three for every
study, so open `lesson1-search` there and put them next to the ones below:

| here | in optuna-dashboard |
|---|---|
| the table of trials | **Trials (Table)**, sortable on the value |
| one panel per axis | **Analytics → Slice** |
| the parallel-coordinates plot | **Smart Selection** |

## 4.2 Reading the result

The temptation is to read off the winning config and move on. Do the opposite: look at how
accuracy varies along **each axis separately**, which tells you whether the axis mattered
at all.

In [ ]:
COLUMNS = ["encoder", "n_hidden_layers", "hidden_dim", "sigma", "lr"]
this_run = [
    t
    for t in study.trials()
    if t.state == optuna.trial.TrialState.COMPLETE
    and t.user_attrs["session"] == SESSION
]
df = pd.DataFrame([{"accuracy": t.value, **t.params} for t in this_run])[
    ["accuracy", *COLUMNS]
]
FORMAT = {"accuracy": "{:.3f}", "sigma": "{:.2f}", "lr": "{:.1e}"}

display(
    df.sort_values("accuracy", ascending=False)
    .head(8)
    .style.format(FORMAT)
    .hide(axis="index")
    .set_caption("the eight best trials of the search")
)

per_encoder = df.groupby("encoder")["accuracy"].agg(
    trials="count", worst="min", best="max"
)
display(
    per_encoder.sort_values("best", ascending=False)
    .style.format({"worst": "{:.3f}", "best": "{:.3f}"})
    .set_caption("per encoder: its worst and its best trial")
)

best_row = df.loc[df["accuracy"].idxmax()]
result_bars(
    {
        f"baseline: linear head on {MODEL}, {SEEDS} seeds": accuracy_per_seed,
        f"best trial of the search: {best_row['encoder']}, "
        f"{int(best_row['n_hidden_layers'])} hidden layers, one run": best_row[
            "accuracy"
        ],
    },
    title="the search against the baseline",
    zoom=True,
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
order = {"encoder": ENCODERS, "n_hidden_layers": [0, 1, 2]}

for ax, col in zip(axes, ["encoder", "n_hidden_layers", "sigma", "lr"]):
    if col in order:
        # categories at fixed positions, in a fixed order, whatever order the trials came in
        position = {value: i for i, value in enumerate(order[col])}
        ax.scatter(df[col].map(position), df["accuracy"], alpha=0.7)
        for value, group in df.groupby(col)["accuracy"]:
            ax.plot(
                [position[value]],
                [group.mean()],
                marker="_",
                ms=40,
                color="crimson",
                mew=2,
            )
        ax.set_xticks(range(len(order[col])), [str(v) for v in order[col]])
        ax.set_xlim(-0.5, len(order[col]) - 0.5)
    else:
        ax.scatter(df[col], df["accuracy"], alpha=0.7)
        if col == "lr":
            ax.set_xscale("log")
            ax.tick_params(axis="x", which="minor", labelbottom=False)  # decades only
    ax.axhline(BASELINE, color="k", ls="--", lw=1)
    ax.axhspan(BASELINE - BASELINE_STD, BASELINE + BASELINE_STD, color="k", alpha=0.08)
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=30 if col == "encoder" else 0)

axes[0].set_ylabel("test accuracy")
fig.suptitle(
    "one axis at a time; red = mean per option, dashed line = the baseline (one Linear on dinov2-small), band = its seed spread"
)
plt.tight_layout()
plt.show()

In [ ]:
from plotting import search_parallel_plot

# The per-axis panels above are good for "does this axis matter at all" -- they are bad at
# showing INTERACTIONS, e.g. "does hidden_dim matter only when n_hidden_layers > 0". A
# parallel-categories plot draws every trial as one line across all axes at once, so a
# pattern that only shows up jointly is still visible. Categorical config values (like
# encoder) plot natively -- no manual code/tickvals mapping needed.
fig = search_parallel_plot(
    df, columns=["encoder", "n_hidden_layers", "hidden_dim", "sigma", "lr", "accuracy"]
)
fig.update_layout(title="Every trial, every axis at once -- colour is accuracy")
fig.show()

Read the four per-axis panels left to right; the parallel-coordinates plot then shows the
same trials as one picture instead of four.

**The encoder panel separates into bands.** The per-encoder table above is the headline:
the *worst* DINOv2 trial in the search beats the *best* resnet-18 trial by more than ten
points, and nothing done to the head closes it. Among the DINOv2 sizes, `dinov2-base` and
`dinov2-large` sit a few points above `dinov2-small`.

**`sigma` and `lr` are clouds.** They scatter trials up and down *within* a band without
organising them. Inside a band much of the visible variation is the spread between runs you
measured in §4.0 — each trial here is a single run.

**`n_hidden_layers` moves less than it looks.** This panel mixes all four encoders, so
the mean of a column says as much about which encoders TPE happened to try there as about
depth. Compare within one encoder in the table of trials instead. On `dinov2-small` a hidden
layer is worth about two points over a single `Linear` when the winner is rerun a few times
(notebook 05 measures it); in this search that effect is a few trials among the
small ones, so it can hide in the spread between runs; on the larger DINOv2 encoders the best trials turn up with and without
one. In the parallel-coordinates plot: the brightest lines run through `dinov2-base` and
`dinov2-large`, then fan out across `n_hidden_layers`, `hidden_dim`, `sigma` and `lr`.

Put the two effects side by side and the answer to "what should I spend my time on" is
plain:

> **The leverage is upstream, in the encoder.** Choosing the vectors moved accuracy by
> more than ten points; everything between the vectors and the loss — the part that feels
> most like "doing machine learning" — moved it by about two.

### Do not over-generalise this

This holds for **a small head on a strong frozen encoder**. It is not a law about neural
architecture, and two later lessons are counter-examples by construction:

- **Lesson 3 (time series)** has no general-purpose encoder worth freezing. You window the
  series and train end to end, and the architecture is back to being your problem.
- **Lesson 6 (self-supervised)** is about *making* the representation instead of renting
  one. That is precisely the place where architecture decides everything.

The transferable lesson is not "architecture never matters". It is: **find out where the
leverage is on your problem before spending your week on the part that feels most like
engineering.**

## 4.3 The fusion arm

Here is an idea with a good story attached. We have two encoders. DINOv2 is strong, resnet
is weak, but they were trained differently, so perhaps they make *different* mistakes. If
resnet knows anything DINOv2 does not, a model that sees both should beat either alone.

That makes a new shape of model. Until now every model was one chain: one tensor in, layer
after layer, one tensor out. A model with two inputs has two **parallel branches** that are
joined into one. Lesson 2 comes back to this shape (a skip connection is a second route
next to a block), so it is worth seeing it once here.

The one design decision is **how the branches are joined**. The drawing below shows the
two usual answers, with the shape on every arrow:

- **concatenate** (top, what `make_fusion` builds and what is drawn from the real model):
  the widths add up, `(B, 384)` and `(B, 512)` become `(B, 896)`. Nothing is lost and the
  widths do not have to match; the `Linear` after it decides how much to use of each.
- **sum** (bottom, drawn for comparison, not trained here): `a + b` only exists when both
  tensors have the same shape, so one branch first needs a `Linear` to bring 512 down to
  384. In vectormesh that per-branch step is `Parallel([nn.Identity(), nn.Linear(512, 384)])`:
  `Parallel` applies one module to each tensor of the pair. Sum is cheaper after the join,
  but it forces both encoders into one shared 384-wide space, and the projection is one
  more thing to train.

In [ ]:
d1, d2 = DIM["dinov2-small"], DIM["resnet-18"]


def make_fusion():
    return Serial(
        [
            Concatenate2D(),  # (B, 384) + (B, 512) -> (B, 896)
            nn.Linear(d1 + d2, N_CLASSES),  # the baseline's single Linear, wider
        ]
    )


# two rows of zeros per encoder are enough to trace the shape after every layer
example_pair = (torch.zeros(2, d1), torch.zeros(2, d2))
caches_in = [
    [Block("dinov2-small\ncache", f"(B, {d1})", "data")],
    [Block("resnet-18\ncache", f"(B, {d2})", "data")],
]

rows = {
    "concatenate: the widths add up  (make_fusion, traced)": [
        caches_in,
        *trace(make_fusion(), example_pair),
        LOSS,
    ],
    f"sum: the widths must match, so resnet's branch is projected to {d1} first": [
        [
            [Block("dinov2-small\ncache", f"(B, {d1})", "data")],
            [
                Block("resnet-18\ncache", f"(B, {d2})", "data"),
                Block(f"Linear\n{d2} → {d1}", f"(B, {d1})", "proj"),
            ],
        ],
        Block("a + b", f"(B, {d1})", "join"),
        Block(f"Linear\n{d1} → {N_CLASSES}", f"(B, {N_CLASSES})"),
        LOSS,
    ],
}
draw_architectures(rows)
plt.show()

`Concatenate2D` takes the pair directly, at the two streams' own widths. There is no
`Parallel` in front of it because neither stream gets a transform of its own before the
join; `Parallel` earns its place only when the branches *do* something, as in the sum
variant. The head after the join is the same single `Linear` as the baseline, so the
second stream is the only thing that differs from §4.0.

**Before you run the cells below, commit to a prediction.** Write it down, sign and rough
size. Does fusing a strong and a weak encoder beat the strong one alone, and by how much?

A prediction you got wrong teaches you more than one you were handed.

### The data: two vectors per image

`join` puts the two caches together on `source_idx`, so every row carries both vectors of
the same image, and the collate function makes two columns instead of one. Each batch is
then a *pair* of tensors, `(B, 384)` and `(B, 512)`.

In [ ]:
def joined_loaders(batch_size: int = 64):
    out = []
    for split in ("train", "test"):
        # join() aligns rows on source_idx and raises if the two caches disagree
        joined = caches["dinov2-small"][split].join(
            caches["resnet-18"][split], on="source_idx"
        )

        col2 = next(c for c in joined.column_names if c.startswith("embed_"))
        collate2 = CollateParallel(
            vec1_col="embed", vec2_col=col2, target_col="onehot", padder=torch.stack
        )
        out.append(
            DataLoader(
                joined.dataset.map(DEFAULT_ONEHOT),
                batch_size=batch_size,
                shuffle=(split == "train"),
                collate_fn=collate2,
            )
        )
    return out


fuse_train, fuse_test = joined_loaders()
(X1, X2), y = next(iter(fuse_train))
print(
    f"two streams per item: {tuple(X1.shape)} and {tuple(X2.shape)}  ->  target {tuple(y.shape)}"
)

### One seed first

Now run it, **one seed first**, the way you would if you were in a hurry.

In [ ]:
torch.manual_seed(0)
one_seed, _ = run(make_fusion(), fuse_train, fuse_test)

result_bars(
    {
        "baseline, seed 0": accuracy_per_seed[0],
        "fusion, seed 0": one_seed,
    },
    title="one run each",
    zoom=True,
    highlight="fusion, seed 0",
)
plt.show()

Whatever sign that came out, **it is not yet a result.** You have one number from one seed.
§4.0 measured how much this task wobbles between seeds; a single run cannot see past that.

### Then every seed

In [ ]:
def seed_spread(
    make_model, train_dl, test_dl, seeds: int = SEEDS
) -> tuple[np.ndarray, float]:
    """Accuracy per seed, and the mean seconds spent in training steps."""
    out, seconds = [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        accuracy, spent = run(make_model(), train_dl, test_dl)
        out.append(accuracy)
        seconds.append(spent)
    return np.array(out), float(np.mean(seconds))


single = np.array(accuracy_per_seed)  # the baseline runs from 4.0
fusion, fusion_seconds = seed_spread(make_fusion, fuse_train, fuse_test)
runs.log(
    name="fusion",
    role="hypothesis",
    results=Scalars(accuracy=fusion.tolist(), seconds=fusion_seconds),
    dataset="dtd",
    params={
        "dataset": "dtd",
        "encoder": "dinov2-small+resnet-18",
        "embed_dim": d1 + d2,
        "head": "concat+linear",
        "n_hidden_layers": 0,
        "sigma": 0.0,
        "lr": 1e-3,
        "epochs": EPOCHS,
    },
    architecture=Architecture.of(make_fusion(), (X1[:2], X2[:2])),
)


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


diff = fusion.mean() - single.mean()
# the standard error of the difference between the two means
pooled = np.sqrt(fusion.var(ddof=1) / len(fusion) + single.var(ddof=1) / len(single))

result_bars(
    {
        f"baseline: dinov2-small, {SEEDS} seeds": single,
        f"fusion: dinov2-small + resnet-18, {SEEDS} seeds": fusion,
    },
    title="mean, standard deviation and every seed",
    zoom=True,
    highlight=f"fusion: dinov2-small + resnet-18, {SEEDS} seeds",
)
plt.show()

display(
    pd.DataFrame(
        {
            "mean": [single.mean(), fusion.mean()],
            "std": [single.std(), fusion.std()],
            "params": [n_params(nn.Linear(d1, N_CLASSES)), n_params(make_fusion())],
        },
        index=["baseline (dinov2-small)", "fusion (both)"],
    ).style.format({"mean": "{:.4f}", "std": "{:.4f}", "params": "{:,}"})
)
display(
    pd.DataFrame(
        {
            "fusion against the baseline": [
                f"{100 * diff:+.2f} pt",
                f"{diff / pooled:+.1f}",
                f"{int((fusion > single.mean()).sum())} of {len(fusion)}",
                f"{fusion.std() / single.std():.1f}x the baseline's",
            ]
        },
        index=[
            "difference of the means",
            "that difference in sigma",
            "seeds where fusion beats the baseline's mean",
            "spread between seeds",
        ],
    )
)

### What the result says

Read the picture and the tables above, not your hopes. On this run, over enough seeds:

**Fusion beats the baseline, and by more than the noise.** About two points, on every
seed, with a seed spread close to the baseline's, for a bit more than twice the
parameters. So resnet-18 *does* carry something that a single `Linear` on `dinov2-small`
does not get out of DINOv2 alone. That is a result, and it stands with its
spread and its sigma.

**Then put it next to §4.2.** The same two points come from giving `dinov2-small` a hidden
layer instead of a second encoder, and the `dinov2-base` and `dinov2-large` trials sit
above the fused model with one encoder each. The fusion win is real; it is not the cheapest
way to that accuracy. A complete conclusion says both.

### One seed agreed this time — you could not have known

Seed 0 on its own put fusion ahead, and the seed spread confirmed it. Do not take the
wrong lesson from that. A difference of one or two points is exactly the size a single
seed can manufacture or hide, and the only way to tell a real difference from a lucky
draw is to run the other seeds. This time the win survived. The habit is the same either
way: measure the spread first, report every difference against it.

### So is fusion good or bad?

Neither, and the question is malformed. What you have measured is that *this* fusion, of
*these* two encoders, with *this* head, on *this* dataset, at *this* training size, is
worth about two points over a linear head. Change any of those and the size, or the sign,
can change with it: give both arms a hidden layer and the gap shrinks; a genuinely
complementary second stream (text alongside images, a different sensor, a different
modality) is a completely different proposition from a weaker view of the same pixels.

A student who leaves this notebook believing "fusion is good" has learned something false
just as surely as one who leaves believing "fusion is bad". What is worth keeping is the
loop:

> have an idea → state it so it could fail → build it cheaply → measure it against a
> baseline you established **first** → report what happened, including how it compares
> with the cheaper alternatives

The cache is what makes that loop cost two seconds. That is the real reason this lesson
started there.

## 4.4 A design in a `.py` file, read with atlas

`make_fusion` is two lines, and the drawing above was enough to read it. A design with
more parts needs more care: you want to see every branch and every tensor shape *before*
you spend training time on it. The way this course does that:

1. Write the model in a `.py` file, with **one method per stage**. A tensor you want a
   shape on gets a name, and the signature of each method says which shape goes in and
   which shape comes out. Inside a stage the vectormesh components compose as above
   (`Parallel`, `Concatenate2D`, a head).
2. Read the file with **atlas**, which draws those stages and the shapes between them.
3. Only then train it.

[`fusion_architecture.py`](fusion_architecture.py), next to this notebook, is a worked
example: a fusion where each branch is first projected to 128 before the two are
concatenated. That projection is a stage the search may leave out: `@option("project")`
marks it, `Config` holds the choice, and `forward` runs it under `if self.config.project:`.
The cell below shows the file. Read `forward` first: it is the whole design in a few
lines, and every name in it is a stage defined above it.

In [ ]:
from IPython.display import Code

Code(filename="fusion_architecture.py", language="python")

Now read it with atlas. In a terminal, from the root of the repository:

```bash
atlas-code-nightly view notebooks/1_representation_and_head/fusion_architecture.py
```

That opens a page in your browser. Click `main`, then `model.forward`, and click a tensor
box to open its shape. You get this:

![fusion_architecture.py in atlas](fusion_atlas.png)

Read it from left to right. A rounded box is a stage (a method), a square box is a tensor
with its shape, written in the widths the model was built with: `{self.shared_dim}` is
the `shared_dim` of `Config`. The pair `streams` goes to `project`, which sits in a
dashed frame, `only when self.config.project`: that is the stage a search may leave out.
`join` makes one tensor of them, `joined_dim` wide, and `classify` ends at `n_classes`.

Running the file builds both subgraphs and prints the same stages with their numbers:

```
project=False   (845,871 params)
  input        (2, 384)
  join         (2, 896)
  classify     (2, 47)

project=True   (192,815 params)
  input        (2, 384)
  project      (2, 128)
  join         (2, 256)
  classify     (2, 47)
```

If a shape is not what you meant, you found the mistake before training anything.

The shapes are also checked while the model runs: `@jaxtyped` on a method raises an error
as soon as a tensor does not have the shape its signature says. So the picture cannot
drift away from the code.

**Then train it**, the same way as the fusion above. The model in the file takes the pair
the joined loader gives, as `make_fusion` does.

In [ ]:
from fusion_architecture import Config, Fusion


def make_projected_fusion():
    return Fusion(
        Config(shared_dim=128), dino_dim=d1, resnet_dim=d2, n_classes=N_CLASSES
    )


projected, projected_seconds = seed_spread(make_projected_fusion, fuse_train, fuse_test)

FROM_FILE = f"fusion_architecture.py: {n_params(make_projected_fusion()):,} weights"
result_bars(
    {
        f"baseline: dinov2-small, {n_params(nn.Linear(d1, N_CLASSES)):,} weights": single,
        f"fusion, concatenate: {n_params(make_fusion()):,} weights": fusion,
        FROM_FILE: projected,
    },
    title=f"the design from the file against the two models above ({SEEDS} seeds each)",
    highlight=FROM_FILE,
    zoom=True,
)
plt.show()

This is the way of working notebook 05 asks of you in Part 2: your own design goes in
[`exercises/designs.py`](exercises/designs.py), you read it with atlas, and
[`exercises/study.py`](exercises/study.py) searches over it.
[`06_your_study.html`](06_your_study.html) shows that loop step by step.

## What you now know

- When a run costs seconds, searching beats guessing — and TPE beats grid search at it.
- Read a search one axis at a time. An axis that does not organise the scatter did not
  matter.
- On a frozen encoder the leverage is upstream: the encoder moves accuracy by more than
  ten points, the head knobs by about two. This is *not* a general law about
  architecture — see L3 and L6.
- Establish the baseline and its seed spread **before** searching, or you cannot tell a
  win from a wobble — and log it, so the comparison is one sort in optuna-dashboard.
- A design with branches goes in a `.py` file with one typed method per stage, and you
  read it with atlas before you train it.
- A result is only complete next to its alternatives: a fusion that wins by two points is
  worth less when one hidden layer, or a bigger encoder, wins by as much.

## Exercises

Notebook 05 is where you design and run your own search. Two short ones first:

**E1 — how many trials do you actually need?** Rerun the search with `N_TRIALS = 8` and
`N_TRIALS = 48`. Does the best config change? Does the *conclusion* change? Which of those
two questions is the one that matters? Compare the trials in optuna-dashboard by their
`session` user attr.

**E2 — change the design in the file.** §4.4 projected both branches to 128 before the
join. Open `fusion_architecture.py` and make the bottleneck much narrower (`shared_dim=16`
in the cell of §4.4 is enough), or add a stage of your own to the class. Read it with
atlas first and check the shapes, then measure it over seeds against `make_fusion`. Does
it still beat the baseline? Explain, in one sentence, where the information went.